## Section 1: Environment Setup and Drive Connection

In [ ]:
# Mount Google Drive to access and save datasets
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import re
import os

Mounted at /content/drive


## Section 2: Dataset Loading, Filtering, and Scaling to 200 Questions

In [ ]:
# Define target diseases and their corresponding keywords for filtering
target_diseases = [
    "Hypertension", "Diabetes", "Asthma", "COPD", "Coronary Heart Disease",
    "Heart Failure", "Chronic Back Pain", "Depression", "Osteoarthritis", "Dementia"
]

disease_keywords = {
    "Hypertension": ['hypertension', 'hypertensive', 'blood pressure is 180', 'blood pressure is 160', 'blood pressure is 170'],
    "Diabetes": ['diabetes', 'hba1c', 'diabetic'],
    "Asthma": ['asthma', 'asthmatic'],
    "COPD": ['copd', 'chronic obstructive', 'emphysema'],
    "Coronary Heart Disease": ['coronary', 'myocardial infarction', 'angina'],
    "Heart Failure": ['heart failure', 'chf', 'cardiac failure', 'heart failing'],
    "Chronic Back Pain": ['back pain', 'lumbar', 'sciatica'],
    "Depression": ['depression', 'depressed', 'major depressive'],
    "Osteoarthritis": ['osteoarthritis', 'joint pain'],
    "Dementia": ['dementia', 'alzheimer', 'memory loss', 'cognitive decline', 'forgetful']
}

def get_disease(text):
    text = str(text).lower()
    for disease, keywords in disease_keywords.items():
        for kw in keywords:
            if kw in text:
                return disease
    return "Other"

# Load the full initial dataset
df_full = pd.read_csv("GP_Top10_Bilingual_Open_EndedQ.csv")

# Map diseases based on English questions
df_full['Disease'] = df_full['English_Question'].apply(get_disease)

# STRICT FILTER: Keep ONLY rows matching the 10 target diseases
df_filtered = df_full[df_full['Disease'].isin(target_diseases)]

# Sample up to 20 questions per disease to build the base dataset
sampled_dfs = []
for disease in target_diseases:
    df_disease = df_filtered[df_filtered['Disease'] == disease]
    # Sample 20 questions (or the max available if fewer than 20 exist)
    sampled_dfs.append(df_disease.sample(min(20, len(df_disease)), random_state=42))

# Combine into the initial 159-question dataframe
df_initial_sample = pd.concat(sampled_dfs)

# Calculate the exact shortfall needed to reach 200
shortfall = 200 - len(df_initial_sample)

if shortfall > 0:
    # Create a pool of remaining unselected questions STRICTLY from the target diseases
    # This prevents any "Other" diseases from leaking into the final dataset
    selected_indices = df_initial_sample.index
    remaining_target_pool = df_filtered.drop(index=selected_indices)

    # Sample the exact amount needed to reach 200 from the surplus of well-represented diseases
    additional_samples = remaining_target_pool.sample(n=shortfall, random_state=42)

    # Combine the base samples with the surplus additions
    df_200 = pd.concat([df_initial_sample, additional_samples]).reset_index(drop=True)
else:
    df_200 = df_initial_sample.reset_index(drop=True)

# Remove empty or unnamed columns
df_200 = df_200.loc[:, ~df_200.columns.str.contains('^Unnamed')]

# Verify the final count and distribution (Total should be exactly 200)
print(f"Total questions compiled: {len(df_200)}\n")
print("Final Distribution per disease:")
print(df_200['Disease'].value_counts())

Total questions compiled: 200

Final Distribution per disease:
Disease
Hypertension              45
Diabetes                  31
Asthma                    23
Depression                22
Chronic Back Pain         20
Coronary Heart Disease    20
Osteoarthritis            13
COPD                      10
Dementia                   9
Heart Failure              7
Name: count, dtype: int64


## Section 3: Ground Truth Answer Extraction

In [ ]:
# Function to extract the correct answer text based on the correct letter
def extract_answer_text(question, correct_letter):
    if pd.isna(question) or pd.isna(correct_letter):
        return ""

    # Regex pattern: \(A\)\s*(.*?)(?=\n\s*\([B-Z]\)|\n\s*\*\*Answer|\Z)
    pattern = rf"\({correct_letter}\)\s*(.*?)(?=\n\s*\([A-Z]\)|\n\s*\*\*Answer|\Z)"
    match = re.search(pattern, question, flags=re.DOTALL | re.IGNORECASE)

    if match:
        return match.group(1).strip().strip('"')

    # Fallback method: string splitting
    try:
        parts = question.split(f"({correct_letter})")
        if len(parts) > 1:
            ans_part = parts[1].split("\n")[0].strip()
            return ans_part.strip('"')
    except:
        pass
    return ""

# Apply extraction for both languages
df_200['English_Correct_Text'] = df_200.apply(lambda row: extract_answer_text(row['English_Question'], row['Correct_Answer']), axis=1)
df_200['German_Correct_Text'] = df_200.apply(lambda row: extract_answer_text(row['German_Question'], row['Correct_Answer']), axis=1)

print("Sample of the extracted answers for verification:")
display(df_200[['Disease', 'Correct_Answer', 'English_Correct_Text', 'German_Correct_Text']].head(5))

Sample of the extracted answers for verification:


,Disease,Correct_Answer,English_Correct_Text,German_Correct_Text
0,Hypertension,D,Blunting of costophrenic angle,Abstumpfung des Costophrenic-Winkels
1,Hypertension,A,Femoropopliteal artery stenosis,Stenose der femoropoplitealen Arterie
2,Hypertension,D,Uterine artery,Uterinarterie
3,Hypertension,C,Contact another family member for consent,Kontaktieren Sie ein anderes Familienmitglied ...
4,Hypertension,D,Surgical gastropexy,Surgical Gastropexie


## Section 4: Formatting and Saving the Final RAG Evaluation Dataset

The final step isolates the specific columns required for the RAG evaluation framework and saves the outputs both locally in the Colab instance and persistently in Google Drive.

In [ ]:
# Select essential columns for Ragas evaluation
final_columns = [
    'Disease',
    'English_Open_Question',
    'English_Correct_Text',
    'German_Open_Question',
    'German_Correct_Text'
]

# Ensure only existing columns are passed to prevent KeyErrors
available_columns = [col for col in final_columns if col in df_200.columns]
df_ragas_200 = df_200[available_columns]

# Define file paths for saving
LOCAL_SAVE_PATH = 'AWMF_Golden_Dataset_200Q_Final.csv'
DRIVE_SAVE_PATH = '/content/drive/MyDrive/AWMF_Golden_Dataset_200Q_Final.csv'

# Save the dataset locally
df_ragas_200.to_csv(LOCAL_SAVE_PATH, index=False)
print(f"Ragas-ready dataset successfully saved locally to: {LOCAL_SAVE_PATH}")

# Save the dataset to Google Drive
try:
    df_ragas_200.to_csv(DRIVE_SAVE_PATH, index=False)
    print(f"Ragas-ready dataset successfully backed up to Google Drive at: {DRIVE_SAVE_PATH}")
except Exception as e:
    print(f"Failed to save to Google Drive. Ensure it is properly mounted. Error: {e}")

# Display final dataset verification
print(f"\nFinal Dataset Shape: {df_ragas_200.shape}")
display(df_ragas_200.head())

Ragas-ready dataset successfully saved locally to: AWMF_Golden_Dataset_200Q_Final.csv
Ragas-ready dataset successfully backed up to Google Drive at: /content/drive/MyDrive/AWMF_Golden_Dataset_200Q_Final.csv

Final Dataset Shape: (200, 5)


,Disease,English_Open_Question,English_Correct_Text,German_Open_Question,German_Correct_Text
0,Hypertension,A 58-year-old man comes to the emergency depar...,Blunting of costophrenic angle,Ein 58-jähriger Mann kommt wegen zunehmender A...,Abstumpfung des Costophrenic-Winkels
1,Hypertension,A 55-year-old man comes to the physician becau...,Femoropopliteal artery stenosis,Ein 55-jähriger Mann kommt wegen einer seit 6 ...,Stenose der femoropoplitealen Arterie
2,Hypertension,A 60-year-old female presents to her gynecolog...,Uterine artery,Eine 60-jährige Frau stellt sich bei ihrem Gyn...,Uterinarterie
3,Hypertension,A 67-year-old man is brought to the emergency ...,Contact another family member for consent,Ein 67-jähriger Mann wird mit plötzlich einset...,Kontaktieren Sie ein anderes Familienmitglied ...
4,Hypertension,A 63-year-old man presents to the ambulatory m...,Surgical gastropexy,Ein 63-jähriger Mann präsentiert sich in der a...,Surgical Gastropexie
